In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install scikit-image scikit-learn -q

In [ ]:
!pip install pandas

In [ ]:
import os
import pandas as pd
import io
from googleapiclient.discovery import build
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authentification
auth.authenticate_user()
creds = GoogleCredentials.get_application_default()
service = build('drive', 'v3', credentials=creds)

# Trouver l'ID du fichier gsheet
results = service.files().list(
    q="name contains 'pipe_detection_label' and trashed=false",
    fields="files(id, name, mimeType)"
).execute()

files = results.get('files', [])
for f in files:
    print(f"Nom: {f['name']}  |  ID: {f['id']}  |  Type: {f['mimeType']}")

Nom: pipe_detection_label.csv  |  ID: 1Sfo5bn9AE34XmZvuKjRhPEj-EKEdlHOr8ZTrsAF73O8  |  Type: application/vnd.google-apps.spreadsheet
Nom: pipe_detection_label  |  ID: 1-d4C7wCPsODEYRAlCk8NS_kPi-401NkI_ZXBZ2X529I  |  Type: application/vnd.google-apps.spreadsheet


In [ ]:
# Exporter le gsheet en CSV directement en mémoire
file_id = "1Sfo5bn9AE34XmZvuKjRhPEj-EKEdlHOr8ZTrsAF73O8"

request = service.files().export_media(
    fileId=file_id,
    mimeType='text/csv'
)

csv_content = request.execute()
df = pd.read_csv(io.BytesIO(csv_content), sep=',')

print(f"✅ CSV chargé : {len(df)} lignes")
print(df['label'].value_counts())
print(df.head(3))

✅ CSV chargé : 4715 lignes
label
1    2829
0    1886
Name: count, dtype: int64
                                      field_file  label coverage_type  \
0  sample_00000_perfect_straight_clean_field.npz      1       perfect   
1  sample_00001_perfect_straight_clean_field.npz      1       perfect   
2  sample_00002_perfect_straight_clean_field.npz      1       perfect   

      shape  noisy pipe_type  
0  straight  False    single  
1  straight  False    single  
2  straight  False    single  


In [ ]:
import os, sys, time, io
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report, confusion_matrix
from skimage.transform import resize
from googleapiclient.discovery import build
from google.colab import auth
from oauth2client.client import GoogleCredentials
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ──────────────────────────────────────────────────────
DATA_DIR   = "/content/drive/MyDrive/skipper_task3/Training_data_inspection_validation_float16"
OUTPUT_DIR = "/content/drive/MyDrive/skipper_task3/task3_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE     = 224
BATCH_SIZE   = 32
NUM_EPOCHS   = 40
LR           = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE     = 8

DEVICE = torch.device("cuda")
print(f"✅ GPU : {torch.cuda.get_device_name(0)}")
print(f"   VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── CHARGEMENT CSV via API Drive (évite le problème .gsheet) ────
print("\n📂 Chargement du CSV depuis Google Drive...")
auth.authenticate_user()
creds   = GoogleCredentials.get_application_default()
service = build('drive', 'v3', credentials=creds)

results = service.files().list(
    q="name contains 'pipe_detection_label' and trashed=false",
    fields="files(id, name, mimeType)"
).execute()

# Prendre le fichier dans skipper_task3 (pas skipper_ndt)
file_id = None
for f in results.get('files', []):
    print(f"  Trouvé : {f['name']}  ({f['mimeType']})")
    if 'gsheet' in f['mimeType'] or f['name'].endswith('.csv'):
        file_id = f['id']

csv_bytes = service.files().export_media(fileId=file_id, mimeType='text/csv').execute()
df = pd.read_csv(io.BytesIO(csv_bytes), sep=',')
print(f"✅ CSV chargé : {len(df)} lignes")
print(f"   Label 1: {(df.label==1).sum()}  |  Label 0: {(df.label==0).sum()}")

# ── ACP ALIGNMENT ───────────────────────────────────────────────
def compute_pca_angle(channel_bz):
    valid_mask = ~np.isnan(channel_bz)
    if valid_mask.sum() < 10:
        return 0.0
    vals = channel_bz[valid_mask]
    threshold = np.percentile(vals, 75)
    signal_mask = (channel_bz > threshold) & valid_mask
    rows, cols = np.where(signal_mask)
    if len(rows) < 10:
        return 0.0
    pts = np.column_stack([cols.astype(float), rows.astype(float)])
    pts -= pts.mean(axis=0)
    cov = np.cov(pts.T)
    if cov.ndim < 2:
        return 0.0
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    dominant = eigenvectors[:, np.argmax(eigenvalues)]
    return np.degrees(np.arctan2(dominant[1], dominant[0]))

def rotate_array(data_4ch, angle_deg):
    from scipy.ndimage import rotate as scipy_rotate
    rotated = np.zeros_like(data_4ch)
    for c in range(4):
        ch = data_4ch[:, :, c].copy()
        ch[np.isnan(ch)] = 0.0
        rotated[:, :, c] = scipy_rotate(ch, angle=-angle_deg, reshape=False, order=1, cval=0.0)
    return rotated

def preprocess_sample(npz_path, img_size=IMG_SIZE):
    try:
        data = np.load(npz_path)['data'].astype(np.float32)
    except:
        return np.zeros((4, img_size, img_size), dtype=np.float32)
    data = np.nan_to_num(data, nan=0.0)
    if data.shape[0] > 30 and data.shape[1] > 30:
        angle = compute_pca_angle(data[:, :, 2])
        if abs(angle) > 5.0 and abs(angle) < 85.0:
            data = rotate_array(data, angle)
    data = resize(data, (img_size, img_size, 4), order=1, preserve_range=True, anti_aliasing=True)
    for c in range(4):
        ch = data[:, :, c]
        data[:, :, c] = (ch - ch.mean()) / (ch.std() + 1e-8)
    return data.transpose(2, 0, 1).astype(np.float32)

# ── DATASET ──────────────────────────────────────────────────────
class PipelineDataset(Dataset):
    def __init__(self, df, data_dir, augment=False):
        self.df       = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.augment  = augment

    def __len__(self):
        return len(self.df)

    def _augment(self, x):
        if np.random.rand() > 0.5:
            x = x[:, :, ::-1].copy()
        if np.random.rand() > 0.5:
            x = x[:, ::-1, :].copy()
        if np.random.rand() > 0.7:
            x = x + np.random.randn(*x.shape).astype(np.float32) * 0.05
        return x

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = os.path.join(self.data_dir, row['field_file'])
        label = int(row['label'])
        x = preprocess_sample(path)
        if self.augment:
            x = self._augment(x)
        return torch.tensor(x, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

# ── CBAM ─────────────────────────────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, in_ch, r=16):
        super().__init__()
        mid = max(1, in_ch // r)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.mx  = nn.AdaptiveMaxPool2d(1)
        self.fc  = nn.Sequential(nn.Flatten(), nn.Linear(in_ch, mid), nn.ReLU(), nn.Linear(mid, in_ch))
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return x * self.sig(self.fc(self.avg(x)) + self.fc(self.mx(x))).unsqueeze(-1).unsqueeze(-1)

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)
        self.sig  = nn.Sigmoid()
    def forward(self, x):
        return x * self.sig(self.conv(torch.cat([x.mean(1,keepdim=True), x.max(1,keepdim=True)[0]], 1)))

class CBAM(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.ca = ChannelAttention(in_ch)
        self.sa = SpatialAttention()
    def forward(self, x):
        return self.sa(self.ca(x))

# ── RESNET18 + CBAM ───────────────────────────────────────────────
class Block(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, cbam=False):
        super().__init__()
        self.c1   = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.b1   = nn.BatchNorm2d(out_ch)
        self.c2   = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.b2   = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.att  = CBAM(out_ch) if cbam else None
        self.down = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_ch)
        ) if stride != 1 or in_ch != out_ch else None

    def forward(self, x):
        r = x
        o = self.relu(self.b1(self.c1(x)))
        o = self.b2(self.c2(o))
        if self.att:  o = self.att(o)
        if self.down: r = self.down(x)
        return self.relu(o + r)

class ResNet18CBAM(nn.Module):
    def __init__(self, in_ch=4, n_cls=2, drop=0.4):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1)
        )
        self.l1 = nn.Sequential(Block(64,64),            Block(64,64))
        self.l2 = nn.Sequential(Block(64,128,stride=2),  Block(128,128))
        self.l3 = nn.Sequential(Block(128,256,stride=2,cbam=True), Block(256,256,cbam=True))
        self.l4 = nn.Sequential(Block(256,512,stride=2,cbam=True), Block(512,512,cbam=True))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Sequential(nn.Dropout(drop), nn.Linear(512, n_cls))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        x = self.stem(x)
        x = self.l1(x); x = self.l2(x)
        x = self.l3(x); x = self.l4(x)
        return self.fc(self.gap(x).flatten(1))

# ── FONCTIONS TRAIN / EVAL ────────────────────────────────────────
def train_epoch(model, loader, opt, crit, device):
    model.train()
    loss_sum, preds_all, labels_all = 0, [], []
    for i, (x, y) in enumerate(loader):
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        out  = model(x)
        loss = crit(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        loss_sum += loss.item()
        preds_all.extend(out.argmax(1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())
        if (i+1) % 50 == 0:
            print(f"    batch {i+1}/{len(loader)}  loss={loss.item():.4f}", flush=True)
    n = len(loader)
    return loss_sum/n, accuracy_score(labels_all, preds_all), recall_score(labels_all, preds_all, zero_division=0)

@torch.no_grad()
def eval_epoch(model, loader, crit, device):
    model.eval()
    loss_sum, preds_all, labels_all = 0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out  = model(x)
        loss_sum += crit(out, y).item()
        preds_all.extend(out.argmax(1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())
    n = len(loader)
    return (loss_sum/n,
            accuracy_score(labels_all, preds_all),
            recall_score(labels_all, preds_all, zero_division=0),
            f1_score(labels_all, preds_all, zero_division=0),
            preds_all, labels_all)

# ── ENTRAÎNEMENT ─────────────────────────────────────────────────
print("\n" + "="*60)
print("TÂCHE 3 — ACP + ResNet18 + CBAM  |  A100 GPU")
print("="*60)

tr_val, df_test  = train_test_split(df, test_size=0.15, stratify=df.label, random_state=42)
df_train, df_val = train_test_split(tr_val, test_size=0.15, stratify=tr_val.label, random_state=42)
print(f"Train: {len(df_train)}  |  Val: {len(df_val)}  |  Test: {len(df_test)}")

n0 = (df_train.label==0).sum()
n1 = (df_train.label==1).sum()
w  = torch.tensor([len(df_train)/(2*n0), len(df_train)/(2*n1)], dtype=torch.float32).to(DEVICE)
print(f"Poids : [w0={w[0]:.3f}, w1={w[1]:.3f}]")

train_dl = DataLoader(PipelineDataset(df_train, DATA_DIR, augment=True),
                      batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(PipelineDataset(df_val,   DATA_DIR, augment=False),
                      batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(PipelineDataset(df_test,  DATA_DIR, augment=False),
                      batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model = ResNet18CBAM().to(DEVICE)
print(f"Paramètres : {sum(p.numel() for p in model.parameters()):,}")

crit  = nn.CrossEntropyLoss(weight=w)
opt   = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = CosineAnnealingLR(opt, T_max=NUM_EPOCHS, eta_min=LR*0.01)

best_f1, best_ep, patience_cnt = 0.0, 0, 0
best_path = os.path.join(OUTPUT_DIR, "task3_best_model.pth")

print(f"\n🚀 Entraînement — {NUM_EPOCHS} epochs max\n" + "-"*60)

for ep in range(1, NUM_EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_acc, tr_rec             = train_epoch(model, train_dl, opt, crit, DEVICE)
    vl_loss, vl_acc, vl_rec, vl_f1,_,_ = eval_epoch(model, val_dl,   crit, DEVICE)
    sched.step()

    print(f"Ep {ep:3d} | "
          f"Tr  Loss={tr_loss:.4f} Acc={tr_acc:.3f} Rec={tr_rec:.3f} | "
          f"Val Loss={vl_loss:.4f} Acc={vl_acc:.3f} Rec={vl_rec:.3f} F1={vl_f1:.3f} | "
          f"{time.time()-t0:.0f}s")

    if vl_f1 > best_f1:
        best_f1, best_ep, patience_cnt = vl_f1, ep, 0
        torch.save({
            'epoch': ep,
            'model_state_dict': model.state_dict(),
            'val_f1': vl_f1, 'val_acc': vl_acc, 'val_rec': vl_rec
        }, best_path)
        print(f"  ✅ Meilleur modèle sauvegardé (F1={vl_f1:.4f})")
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"\n⏹️  Early stopping à l'epoch {ep}")
            break

print(f"\n🏆 Meilleur : Epoch {best_ep}  F1={best_f1:.4f}")

# ── ÉVALUATION FINALE ─────────────────────────────────────────────
model.load_state_dict(torch.load(best_path, map_location=DEVICE)['model_state_dict'])
_, t_acc, t_rec, t_f1, t_preds, t_labels = eval_epoch(model, test_dl, crit, DEVICE)

print("\n" + "="*60)
print("RÉSULTATS FINAUX SUR LE TEST SET")
print("="*60)
print(f"Accuracy : {t_acc*100:.2f}%  [objectif >90%]  {'✅' if t_acc>=0.90 else '❌'}")
print(f"Recall   : {t_rec*100:.2f}%  [objectif >85%]  {'✅' if t_rec>=0.85 else '❌'}")
print(f"F1-Score : {t_f1:.4f}")
print()
print(classification_report(t_labels, t_preds, target_names=['Non détectable','Détectable']))
cm = confusion_matrix(t_labels, t_preds)
print(f"Matrice de confusion :")
print(f"  TN={cm[0,0]}  FP={cm[0,1]}")
print(f"  FN={cm[1,0]}  TP={cm[1,1]}")
print(f"\n💾 Modèle sauvegardé : {best_path}")

✅ GPU : NVIDIA A100-SXM4-80GB
   VRAM : 85.1 GB

📂 Chargement du CSV depuis Google Drive...
  Trouvé : pipe_detection_label.csv  (application/vnd.google-apps.spreadsheet)
  Trouvé : pipe_detection_label  (application/vnd.google-apps.spreadsheet)
✅ CSV chargé : 4715 lignes
   Label 1: 2829  |  Label 0: 1886

TÂCHE 3 — ACP + ResNet18 + CBAM  |  A100 GPU
Train: 3405  |  Val: 602  |  Test: 708
Poids : [w0=1.250, w1=0.833]
Paramètres : 11,264,618

🚀 Entraînement — 40 epochs max
------------------------------------------------------------
    batch 50/107  loss=0.6620


KeyboardInterrupt: 

In [ ]:
import os, sys, time, io
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report, confusion_matrix
from skimage.transform import resize
from scipy.ndimage import rotate as scipy_rotate
from googleapiclient.discovery import build
from google.colab import auth
from oauth2client.client import GoogleCredentials
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ──────────────────────────────────────────────────────
DATA_DIR   = "/content/drive/MyDrive/skipper_task3/Training_data_inspection_validation_float16"
OUTPUT_DIR = "/content/drive/MyDrive/skipper_task3/task3_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE     = 224
BATCH_SIZE   = 64   # On peut monter car plus de goulot CPU
NUM_EPOCHS   = 40
LR           = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE     = 8

DEVICE = torch.device("cuda")
print(f"✅ GPU : {torch.cuda.get_device_name(0)}")
print(f"   VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── CHARGEMENT CSV ───────────────────────────────────────────────
print("\n📂 Chargement CSV...")
auth.authenticate_user()
creds   = GoogleCredentials.get_application_default()
service = build('drive', 'v3', credentials=creds)
results = service.files().list(
    q="name contains 'pipe_detection_label' and trashed=false",
    fields="files(id, name, mimeType)"
).execute()
file_id = None
for f in results.get('files', []):
    if 'gsheet' in f['mimeType'] or f['name'].endswith('.csv'):
        file_id = f['id']
        print(f"  Utilisation : {f['name']}")
        break
csv_bytes = service.files().export_media(fileId=file_id, mimeType='text/csv').execute()
df = pd.read_csv(io.BytesIO(csv_bytes), sep=',')
print(f"✅ {len(df)} échantillons chargés")

# ── ACP ALIGNMENT ───────────────────────────────────────────────
def compute_pca_angle(channel_bz):
    valid_mask = ~np.isnan(channel_bz)
    if valid_mask.sum() < 10:
        return 0.0
    vals = channel_bz[valid_mask]
    threshold = np.percentile(vals, 75)
    signal_mask = (channel_bz > threshold) & valid_mask
    rows, cols = np.where(signal_mask)
    if len(rows) < 10:
        return 0.0
    pts = np.column_stack([cols.astype(float), rows.astype(float)])
    pts -= pts.mean(axis=0)
    cov = np.cov(pts.T)
    if cov.ndim < 2:
        return 0.0
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    dominant = eigenvectors[:, np.argmax(eigenvalues)]
    return np.degrees(np.arctan2(dominant[1], dominant[0]))

def preprocess_one(npz_path, img_size=IMG_SIZE):
    try:
        data = np.load(npz_path)['data'].astype(np.float32)
    except:
        return np.zeros((4, img_size, img_size), dtype=np.float32)
    data = np.nan_to_num(data, nan=0.0)
    # ACP alignment
    if data.shape[0] > 30 and data.shape[1] > 30:
        angle = compute_pca_angle(data[:, :, 2])
        if abs(angle) > 5.0 and abs(angle) < 85.0:
            rotated = np.zeros_like(data)
            for c in range(4):
                rotated[:, :, c] = scipy_rotate(data[:, :, c], angle=-angle,
                                                 reshape=False, order=1, cval=0.0)
            data = rotated
    # Resize
    data = resize(data, (img_size, img_size, 4), order=1,
                  preserve_range=True, anti_aliasing=True)
    # Normalisation
    for c in range(4):
        ch = data[:, :, c]
        data[:, :, c] = (ch - ch.mean()) / (ch.std() + 1e-8)
    return data.transpose(2, 0, 1).astype(np.float32)

# ── PRÉCALCUL COMPLET EN RAM ─────────────────────────────────────
# C'est la clé : on fait tout le preprocessing UNE SEULE FOIS
# Avec High RAM Colab (~50GB RAM CPU) on peut tout stocker
print("\n⚡ Précalcul du preprocessing pour les 4715 samples...")
print("   (fait une seule fois, ~10-15 min, puis entraînement ultra-rapide)\n")

all_X = np.zeros((len(df), 4, IMG_SIZE, IMG_SIZE), dtype=np.float32)
all_y = df['label'].values.astype(np.int64)

t_start = time.time()
for i, row in tqdm(df.iterrows(), total=len(df), desc="Preprocessing"):
    path = os.path.join(DATA_DIR, row['field_file'])
    all_X[i] = preprocess_one(path)

elapsed = time.time() - t_start
print(f"\n✅ Preprocessing terminé en {elapsed/60:.1f} minutes")
print(f"   Taille en RAM : {all_X.nbytes / 1e9:.2f} GB")

# Convertir en tensors PyTorch
X_tensor = torch.tensor(all_X, dtype=torch.float32)
y_tensor = torch.tensor(all_y, dtype=torch.long)
del all_X  # Libérer la RAM numpy

# ── SPLIT ────────────────────────────────────────────────────────
indices = np.arange(len(df))
idx_trval, idx_test = train_test_split(indices, test_size=0.15,
                                        stratify=all_y, random_state=42)
idx_train, idx_val  = train_test_split(idx_trval, test_size=0.15,
                                        stratify=all_y[idx_trval], random_state=42)

print(f"\nSplit → Train: {len(idx_train)} | Val: {len(idx_val)} | Test: {len(idx_test)}")

# TensorDataset = accès instantané, zéro latence CPU
train_ds = TensorDataset(X_tensor[idx_train], y_tensor[idx_train])
val_ds   = TensorDataset(X_tensor[idx_val],   y_tensor[idx_val])
test_ds  = TensorDataset(X_tensor[idx_test],  y_tensor[idx_test])

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=0, pin_memory=False)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=0, pin_memory=False)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=0, pin_memory=False)

# ── AUGMENTATION À LA VOLÉE (légère, sur GPU) ────────────────────
def augment_batch(x):
    """Augmentation rapide directement sur tensors GPU."""
    # Flip horizontal
    mask_h = torch.rand(x.size(0), device=x.device) > 0.5
    x[mask_h] = x[mask_h].flip(-1)
    # Flip vertical
    mask_v = torch.rand(x.size(0), device=x.device) > 0.5
    x[mask_v] = x[mask_v].flip(-2)
    # Bruit gaussien
    mask_n = torch.rand(x.size(0), device=x.device) > 0.7
    if mask_n.any():
        x[mask_n] += torch.randn_like(x[mask_n]) * 0.05
    return x

# ── CBAM ─────────────────────────────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, in_ch, r=16):
        super().__init__()
        mid = max(1, in_ch // r)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.mx  = nn.AdaptiveMaxPool2d(1)
        self.fc  = nn.Sequential(nn.Flatten(), nn.Linear(in_ch, mid),
                                 nn.ReLU(), nn.Linear(mid, in_ch))
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return x * self.sig(self.fc(self.avg(x)) + self.fc(self.mx(x))).unsqueeze(-1).unsqueeze(-1)

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)
        self.sig  = nn.Sigmoid()
    def forward(self, x):
        return x * self.sig(self.conv(
            torch.cat([x.mean(1,keepdim=True), x.max(1,keepdim=True)[0]], 1)))

class CBAM(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.ca = ChannelAttention(in_ch)
        self.sa = SpatialAttention()
    def forward(self, x):
        return self.sa(self.ca(x))

# ── RESNET18 + CBAM ───────────────────────────────────────────────
class Block(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, cbam=False):
        super().__init__()
        self.c1   = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.b1   = nn.BatchNorm2d(out_ch)
        self.c2   = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.b2   = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.att  = CBAM(out_ch) if cbam else None
        self.down = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_ch)
        ) if stride != 1 or in_ch != out_ch else None
    def forward(self, x):
        r = x
        o = self.relu(self.b1(self.c1(x)))
        o = self.b2(self.c2(o))
        if self.att:  o = self.att(o)
        if self.down: r = self.down(x)
        return self.relu(o + r)

class ResNet18CBAM(nn.Module):
    def __init__(self, in_ch=4, n_cls=2, drop=0.4):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1))
        self.l1 = nn.Sequential(Block(64,64),           Block(64,64))
        self.l2 = nn.Sequential(Block(64,128,stride=2), Block(128,128))
        self.l3 = nn.Sequential(Block(128,256,stride=2,cbam=True), Block(256,256,cbam=True))
        self.l4 = nn.Sequential(Block(256,512,stride=2,cbam=True), Block(512,512,cbam=True))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Sequential(nn.Dropout(drop), nn.Linear(512, n_cls))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    def forward(self, x):
        x = self.stem(x)
        x = self.l1(x); x = self.l2(x)
        x = self.l3(x); x = self.l4(x)
        return self.fc(self.gap(x).flatten(1))

# ── TRAIN / EVAL ──────────────────────────────────────────────────
def train_epoch(model, loader, opt, crit, device):
    model.train()
    loss_sum, preds_all, labels_all = 0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        x = augment_batch(x)          # augmentation sur GPU, instantané
        opt.zero_grad()
        loss = crit(model(x), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        loss_sum += loss.item()
        preds_all.extend(model(x).argmax(1).detach().cpu().numpy())
        labels_all.extend(y.cpu().numpy())
    n = len(loader)
    return (loss_sum/n,
            accuracy_score(labels_all, preds_all),
            recall_score(labels_all, preds_all, zero_division=0))

@torch.no_grad()
def eval_epoch(model, loader, crit, device):
    model.eval()
    loss_sum, preds_all, labels_all = 0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss_sum += crit(out, y).item()
        preds_all.extend(out.argmax(1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())
    n = len(loader)
    return (loss_sum/n,
            accuracy_score(labels_all, preds_all),
            recall_score(labels_all, preds_all, zero_division=0),
            f1_score(labels_all, preds_all, zero_division=0),
            preds_all, labels_all)

# ── ENTRAÎNEMENT ─────────────────────────────────────────────────
n0 = (y_tensor[idx_train] == 0).sum().item()
n1 = (y_tensor[idx_train] == 1).sum().item()
w  = torch.tensor([len(idx_train)/(2*n0), len(idx_train)/(2*n1)],
                   dtype=torch.float32).to(DEVICE)
print(f"Poids : [w0={w[0]:.3f}, w1={w[1]:.3f}]")

model = ResNet18CBAM().to(DEVICE)
print(f"Paramètres : {sum(p.numel() for p in model.parameters()):,}")

crit  = nn.CrossEntropyLoss(weight=w)
opt   = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = CosineAnnealingLR(opt, T_max=NUM_EPOCHS, eta_min=LR*0.01)

best_f1, best_ep, patience_cnt = 0.0, 0, 0
best_path = os.path.join(OUTPUT_DIR, "task3_best_model.pth")

print(f"\n🚀 Entraînement — données en RAM, augmentation GPU")
print(f"   Epoch attendue : ~15-30 secondes\n" + "-"*60)

for ep in range(1, NUM_EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_acc, tr_rec             = train_epoch(model, train_dl, opt, crit, DEVICE)
    vl_loss, vl_acc, vl_rec, vl_f1,_,_ = eval_epoch(model, val_dl, crit, DEVICE)
    sched.step()
    print(f"Ep {ep:3d} | "
          f"Tr Loss={tr_loss:.4f} Acc={tr_acc:.3f} Rec={tr_rec:.3f} | "
          f"Val Loss={vl_loss:.4f} Acc={vl_acc:.3f} Rec={vl_rec:.3f} F1={vl_f1:.3f} | "
          f"{time.time()-t0:.0f}s")
    if vl_f1 > best_f1:
        best_f1, best_ep, patience_cnt = vl_f1, ep, 0
        torch.save({'epoch':ep, 'model_state_dict':model.state_dict(),
                    'val_f1':vl_f1, 'val_acc':vl_acc, 'val_rec':vl_rec}, best_path)
        print(f"  ✅ Sauvegardé (F1={vl_f1:.4f})")
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"\n⏹️  Early stopping epoch {ep}"); break

print(f"\n🏆 Meilleur : Epoch {best_ep}  F1={best_f1:.4f}")

# ── TEST FINAL ────────────────────────────────────────────────────
model.load_state_dict(torch.load(best_path, map_location=DEVICE)['model_state_dict'])
_, t_acc, t_rec, t_f1, t_preds, t_labels = eval_epoch(model, test_dl, crit, DEVICE)

print("\n" + "="*60)
print("RÉSULTATS FINAUX")
print("="*60)
print(f"Accuracy : {t_acc*100:.2f}%  {'✅' if t_acc>=0.90 else '❌'}  [objectif >90%]")
print(f"Recall   : {t_rec*100:.2f}%  {'✅' if t_rec>=0.85 else '❌'}  [objectif >85%]")
print(f"F1-Score : {t_f1:.4f}")
print()
print(classification_report(t_labels, t_preds,
                             target_names=['Non détectable','Détectable']))
cm = confusion_matrix(t_labels, t_preds)
print(f"Matrice confusion : TN={cm[0,0]} FP={cm[0,1]} | FN={cm[1,0]} TP={cm[1,1]}")
print(f"\n💾 Modèle : {best_path}")

✅ GPU : NVIDIA A100-SXM4-80GB
   VRAM : 85.1 GB

📂 Chargement CSV...
  Utilisation : pipe_detection_label.csv
✅ 4715 échantillons chargés

⚡ Précalcul du preprocessing pour les 4715 samples...
   (fait une seule fois, ~10-15 min, puis entraînement ultra-rapide)



Preprocessing: 100%|██████████| 4715/4715 [1:02:50<00:00,  1.25it/s]



✅ Preprocessing terminé en 62.8 minutes
   Taille en RAM : 3.79 GB

Split → Train: 3405 | Val: 602 | Test: 708
Poids : [w0=1.250, w1=0.833]
Paramètres : 11,264,618

🚀 Entraînement — données en RAM, augmentation GPU
   Epoch attendue : ~15-30 secondes
------------------------------------------------------------
Ep   1 | Tr Loss=0.6542 Acc=0.652 Rec=0.647 | Val Loss=0.6402 Acc=0.648 Rec=0.576 F1=0.662 | 5s
  ✅ Sauvegardé (F1=0.6624)
Ep   2 | Tr Loss=0.5552 Acc=0.745 Rec=0.817 | Val Loss=0.5243 Acc=0.741 Rec=0.729 F1=0.771 | 5s
  ✅ Sauvegardé (F1=0.7713)
Ep   3 | Tr Loss=0.4902 Acc=0.783 Rec=0.840 | Val Loss=0.4883 Acc=0.777 Rec=0.903 F1=0.830 | 5s
  ✅ Sauvegardé (F1=0.8295)
Ep   4 | Tr Loss=0.4666 Acc=0.797 Rec=0.833 | Val Loss=0.4696 Acc=0.802 Rec=0.867 F1=0.840 | 5s
  ✅ Sauvegardé (F1=0.8403)
Ep   5 | Tr Loss=0.4134 Acc=0.826 Rec=0.869 | Val Loss=0.4304 Acc=0.787 Rec=0.823 F1=0.823 | 5s
Ep   6 | Tr Loss=0.3930 Acc=0.837 Rec=0.863 | Val Loss=0.4276 Acc=0.826 Rec=0.939 F1=0.866 | 5s
  ✅

In [ ]:
# Sauvegarder les tensors preprocessés sur Drive
print("💾 Sauvegarde des tensors preprocessés...")
torch.save({
    'X': X_tensor,
    'y': y_tensor
}, "/content/drive/MyDrive/skipper_task3/preprocessed_data.pt")
print("✅ Sauvegardé ! La prochaine fois tu chargeras directement ce fichier.")

💾 Sauvegarde des tensors preprocessés...
✅ Sauvegardé ! La prochaine fois tu chargeras directement ce fichier.


In [ ]:
# Charger directement sans recalculer l'ACP
print("⚡ Chargement des données preprocessées...")
checkpoint = torch.load("/content/drive/MyDrive/skipper_task3/preprocessed_data.pt")
X_tensor = checkpoint['X']
y_tensor = checkpoint['y']
print(f"✅ Chargé en quelques secondes — {X_tensor.shape}")

⚡ Chargement des données preprocessées...
✅ Chargé en quelques secondes — torch.Size([4715, 4, 224, 224])


In [ ]:
import os, time, io
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = "/content/drive/MyDrive/skipper_task3/task3_outputs"
DEVICE     = torch.device("cuda")
BATCH_SIZE = 64
NUM_EPOCHS = 40
LR         = 1e-4
PATIENCE   = 8

# ── Chargement direct depuis le .pt sauvegardé ──────────────────
print("⚡ Chargement données preprocessées...")
ckpt     = torch.load("/content/drive/MyDrive/skipper_task3/preprocessed_data.pt")
X_tensor = ckpt['X']
y_tensor = ckpt['y']
print(f"✅ {X_tensor.shape[0]} samples chargés en {X_tensor.shape} — instantané !")

# ── Split identique (même random_state=42) ──────────────────────
indices  = np.arange(len(y_tensor))
all_y    = y_tensor.numpy()
idx_trval, idx_test  = train_test_split(indices, test_size=0.15,
                                         stratify=all_y, random_state=42)
idx_train, idx_val   = train_test_split(idx_trval, test_size=0.15,
                                         stratify=all_y[idx_trval], random_state=42)

train_dl = DataLoader(TensorDataset(X_tensor[idx_train], y_tensor[idx_train]),
                      batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(TensorDataset(X_tensor[idx_val],   y_tensor[idx_val]),
                      batch_size=BATCH_SIZE, shuffle=False)
test_dl  = DataLoader(TensorDataset(X_tensor[idx_test],  y_tensor[idx_test]),
                      batch_size=BATCH_SIZE, shuffle=False)

# ── Poids équilibrés ─────────────────────────────────────────────
# On enlève le déséquilibre de poids — c'était la cause des 72 FP
w = torch.tensor([1.0, 1.0], dtype=torch.float32).to(DEVICE)
print(f"Poids : [w0={w[0]:.2f}, w1={w[1]:.2f}]  ← équilibrés cette fois")

# ── Même architecture ────────────────────────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, in_ch, r=16):
        super().__init__()
        mid = max(1, in_ch // r)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.mx  = nn.AdaptiveMaxPool2d(1)
        self.fc  = nn.Sequential(nn.Flatten(), nn.Linear(in_ch, mid),
                                 nn.ReLU(), nn.Linear(mid, in_ch))
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return x * self.sig(self.fc(self.avg(x)) + self.fc(self.mx(x))).unsqueeze(-1).unsqueeze(-1)

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)
        self.sig  = nn.Sigmoid()
    def forward(self, x):
        return x * self.sig(self.conv(
            torch.cat([x.mean(1,keepdim=True), x.max(1,keepdim=True)[0]], 1)))

class CBAM(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.ca = ChannelAttention(in_ch)
        self.sa = SpatialAttention()
    def forward(self, x):
        return self.sa(self.ca(x))

class Block(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, cbam=False):
        super().__init__()
        self.c1   = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.b1   = nn.BatchNorm2d(out_ch)
        self.c2   = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.b2   = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.att  = CBAM(out_ch) if cbam else None
        self.down = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_ch)
        ) if stride != 1 or in_ch != out_ch else None
    def forward(self, x):
        r = x
        o = self.relu(self.b1(self.c1(x)))
        o = self.b2(self.c2(o))
        if self.att:  o = self.att(o)
        if self.down: r = self.down(x)
        return self.relu(o + r)

class ResNet18CBAM(nn.Module):
    def __init__(self, in_ch=4, n_cls=2, drop=0.4):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1))
        self.l1 = nn.Sequential(Block(64,64),           Block(64,64))
        self.l2 = nn.Sequential(Block(64,128,stride=2), Block(128,128))
        self.l3 = nn.Sequential(Block(128,256,stride=2,cbam=True), Block(256,256,cbam=True))
        self.l4 = nn.Sequential(Block(256,512,stride=2,cbam=True), Block(512,512,cbam=True))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Sequential(nn.Dropout(drop), nn.Linear(512, n_cls))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    def forward(self, x):
        x = self.stem(x)
        x = self.l1(x); x = self.l2(x)
        x = self.l3(x); x = self.l4(x)
        return self.fc(self.gap(x).flatten(1))

def augment_batch(x):
    mask_h = torch.rand(x.size(0), device=x.device) > 0.5
    x[mask_h] = x[mask_h].flip(-1)
    mask_v = torch.rand(x.size(0), device=x.device) > 0.5
    x[mask_v] = x[mask_v].flip(-2)
    if torch.rand(1) > 0.7:
        x += torch.randn_like(x) * 0.05
    return x

def train_epoch(model, loader, opt, crit, device):
    model.train()
    loss_sum, preds_all, labels_all = 0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        x = augment_batch(x)
        opt.zero_grad()
        out  = model(x)
        loss = crit(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        loss_sum += loss.item()
        preds_all.extend(out.argmax(1).detach().cpu().numpy())
        labels_all.extend(y.cpu().numpy())
    n = len(loader)
    return (loss_sum/n,
            accuracy_score(labels_all, preds_all),
            recall_score(labels_all, preds_all, zero_division=0))

@torch.no_grad()
def eval_epoch(model, loader, crit, device):
    model.eval()
    loss_sum, preds_all, labels_all = 0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out  = model(x)
        loss_sum += crit(out, y).item()
        preds_all.extend(out.argmax(1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())
    n = len(loader)
    return (loss_sum/n,
            accuracy_score(labels_all, preds_all),
            recall_score(labels_all, preds_all, zero_division=0),
            f1_score(labels_all, preds_all, zero_division=0),
            preds_all, labels_all)

# ── Entraînement ─────────────────────────────────────────────────
model = ResNet18CBAM().to(DEVICE)
crit  = nn.CrossEntropyLoss(weight=w)
opt   = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = CosineAnnealingLR(opt, T_max=NUM_EPOCHS, eta_min=LR*0.01)

best_f1, best_ep, patience_cnt = 0.0, 0, 0
best_path = os.path.join(OUTPUT_DIR, "task3_best_model_v2.pth")

print(f"\n🚀 Entraînement v2 — poids équilibrés\n" + "-"*60)

for ep in range(1, NUM_EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_acc, tr_rec             = train_epoch(model, train_dl, opt, crit, DEVICE)
    vl_loss, vl_acc, vl_rec, vl_f1,_,_ = eval_epoch(model, val_dl, crit, DEVICE)
    sched.step()
    print(f"Ep {ep:3d} | "
          f"Tr Loss={tr_loss:.4f} Acc={tr_acc:.3f} Rec={tr_rec:.3f} | "
          f"Val Loss={vl_loss:.4f} Acc={vl_acc:.3f} Rec={vl_rec:.3f} F1={vl_f1:.3f} | "
          f"{time.time()-t0:.0f}s")
    if vl_f1 > best_f1:
        best_f1, best_ep, patience_cnt = vl_f1, ep, 0
        torch.save({'epoch':ep, 'model_state_dict':model.state_dict(),
                    'val_f1':vl_f1, 'val_acc':vl_acc, 'val_rec':vl_rec}, best_path)
        print(f"  ✅ Sauvegardé (F1={vl_f1:.4f})")
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"\n⏹️  Early stopping epoch {ep}"); break

# ── Test final ───────────────────────────────────────────────────
model.load_state_dict(torch.load(best_path, map_location=DEVICE)['model_state_dict'])
_, t_acc, t_rec, t_f1, t_preds, t_labels = eval_epoch(model, test_dl, crit, DEVICE)

print("\n" + "="*60)
print("RÉSULTATS FINAUX v2")
print("="*60)
print(f"Accuracy : {t_acc*100:.2f}%  {'✅' if t_acc>=0.90 else '❌'}  [objectif >90%]")
print(f"Recall   : {t_rec*100:.2f}%  {'✅' if t_rec>=0.85 else '❌'}  [objectif >85%]")
print(f"F1-Score : {t_f1:.4f}")
print()
print(classification_report(t_labels, t_preds,
                             target_names=['Non détectable','Détectable']))
cm = confusion_matrix(t_labels, t_preds)
print(f"Matrice confusion : TN={cm[0,0]} FP={cm[0,1]} | FN={cm[1,0]} TP={cm[1,1]}")

⚡ Chargement données preprocessées...
✅ 4715 samples chargés en torch.Size([4715, 4, 224, 224]) — instantané !
Poids : [w0=1.00, w1=1.00]  ← équilibrés cette fois

🚀 Entraînement v2 — poids équilibrés
------------------------------------------------------------
Ep   1 | Tr Loss=0.6265 Acc=0.661 Rec=0.866 | Val Loss=0.5676 Acc=0.738 Rec=0.861 F1=0.797 | 4s
  ✅ Sauvegardé (F1=0.7974)
Ep   2 | Tr Loss=0.5118 Acc=0.751 Rec=0.883 | Val Loss=0.5021 Acc=0.756 Rec=0.867 F1=0.810 | 5s
  ✅ Sauvegardé (F1=0.8098)
Ep   3 | Tr Loss=0.4668 Acc=0.779 Rec=0.900 | Val Loss=0.4636 Acc=0.794 Rec=0.884 F1=0.837 | 5s
  ✅ Sauvegardé (F1=0.8373)
Ep   4 | Tr Loss=0.4171 Acc=0.810 Rec=0.916 | Val Loss=0.4660 Acc=0.791 Rec=0.817 F1=0.824 | 4s
Ep   5 | Tr Loss=0.3726 Acc=0.834 Rec=0.914 | Val Loss=0.4170 Acc=0.792 Rec=0.823 F1=0.826 | 5s
Ep   6 | Tr Loss=0.3430 Acc=0.851 Rec=0.925 | Val Loss=0.4231 Acc=0.816 Rec=0.950 F1=0.861 | 5s
  ✅ Sauvegardé (F1=0.8607)
Ep   7 | Tr Loss=0.3334 Acc=0.852 Rec=0.914 | Val Loss

In [ ]:
import os, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = "/content/drive/MyDrive/skipper_task3/task3_outputs"
DEVICE     = torch.device("cuda")
BATCH_SIZE = 64
NUM_EPOCHS = 50
LR         = 3e-5   # ← réduit, le modèle overfittait trop
PATIENCE   = 10

# ── Chargement instantané ────────────────────────────────────────
ckpt     = torch.load("/content/drive/MyDrive/skipper_task3/preprocessed_data.pt")
X_tensor = ckpt['X']
y_tensor = ckpt['y']
print(f"✅ {X_tensor.shape[0]} samples chargés instantanément")

indices = np.arange(len(y_tensor))
all_y   = y_tensor.numpy()
idx_trval, idx_test  = train_test_split(indices, test_size=0.15,
                                         stratify=all_y, random_state=42)
idx_train, idx_val   = train_test_split(idx_trval, test_size=0.15,
                                         stratify=all_y[idx_trval], random_state=42)

train_dl = DataLoader(TensorDataset(X_tensor[idx_train], y_tensor[idx_train]),
                      batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(TensorDataset(X_tensor[idx_val],   y_tensor[idx_val]),
                      batch_size=BATCH_SIZE, shuffle=False)
test_dl  = DataLoader(TensorDataset(X_tensor[idx_test],  y_tensor[idx_test]),
                      batch_size=BATCH_SIZE, shuffle=False)

# ── Poids intermédiaires ─────────────────────────────────────────
w = torch.tensor([1.5, 0.7], dtype=torch.float32).to(DEVICE)
print(f"Poids : [w0={w[0]:.2f}, w1={w[1]:.2f}]  ← intermédiaires")

# ── Architecture avec dropout renforcé ───────────────────────────
class ChannelAttention(nn.Module):
    def __init__(self, in_ch, r=16):
        super().__init__()
        mid = max(1, in_ch // r)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.mx  = nn.AdaptiveMaxPool2d(1)
        self.fc  = nn.Sequential(nn.Flatten(), nn.Linear(in_ch, mid),
                                 nn.ReLU(), nn.Linear(mid, in_ch))
        self.sig = nn.Sigmoid()
    def forward(self, x):
        return x * self.sig(self.fc(self.avg(x)) + self.fc(self.mx(x))).unsqueeze(-1).unsqueeze(-1)

class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)
        self.sig  = nn.Sigmoid()
    def forward(self, x):
        return x * self.sig(self.conv(
            torch.cat([x.mean(1,keepdim=True), x.max(1,keepdim=True)[0]], 1)))

class CBAM(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.ca = ChannelAttention(in_ch)
        self.sa = SpatialAttention()
    def forward(self, x):
        return self.sa(self.ca(x))

class Block(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, cbam=False):
        super().__init__()
        self.c1   = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.b1   = nn.BatchNorm2d(out_ch)
        self.c2   = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.b2   = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.att  = CBAM(out_ch) if cbam else None
        self.down = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_ch)
        ) if stride != 1 or in_ch != out_ch else None
    def forward(self, x):
        r = x
        o = self.relu(self.b1(self.c1(x)))
        o = self.b2(self.c2(o))
        if self.att:  o = self.att(o)
        if self.down: r = self.down(x)
        return self.relu(o + r)

class ResNet18CBAM(nn.Module):
    def __init__(self, in_ch=4, n_cls=2, drop=0.5):  # ← dropout 0.5
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1))
        self.l1 = nn.Sequential(Block(64,64),           Block(64,64))
        self.l2 = nn.Sequential(Block(64,128,stride=2), Block(128,128))
        self.l3 = nn.Sequential(Block(128,256,stride=2,cbam=True), Block(256,256,cbam=True))
        self.l4 = nn.Sequential(Block(256,512,stride=2,cbam=True), Block(512,512,cbam=True))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(512, 128),   # ← couche intermédiaire ajoutée
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_cls)
        )
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
    def forward(self, x):
        x = self.stem(x)
        x = self.l1(x); x = self.l2(x)
        x = self.l3(x); x = self.l4(x)
        return self.fc(self.gap(x).flatten(1))

def augment_batch(x):
    mask_h = torch.rand(x.size(0), device=x.device) > 0.5
    x[mask_h] = x[mask_h].flip(-1)
    mask_v = torch.rand(x.size(0), device=x.device) > 0.5
    x[mask_v] = x[mask_v].flip(-2)
    if torch.rand(1) > 0.7:
        x += torch.randn_like(x) * 0.05
    return x

def train_epoch(model, loader, opt, crit, device):
    model.train()
    loss_sum, preds_all, labels_all = 0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        x = augment_batch(x)
        opt.zero_grad()
        out  = model(x)
        loss = crit(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        loss_sum += loss.item()
        preds_all.extend(out.argmax(1).detach().cpu().numpy())
        labels_all.extend(y.cpu().numpy())
    n = len(loader)
    return (loss_sum/n,
            accuracy_score(labels_all, preds_all),
            recall_score(labels_all, preds_all, zero_division=0))

@torch.no_grad()
def eval_epoch(model, loader, crit, device):
    model.eval()
    loss_sum, preds_all, labels_all = 0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out  = model(x)
        loss_sum += crit(out, y).item()
        preds_all.extend(out.argmax(1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())
    n = len(loader)
    return (loss_sum/n,
            accuracy_score(labels_all, preds_all),
            recall_score(labels_all, preds_all, zero_division=0),
            f1_score(labels_all, preds_all, zero_division=0),
            preds_all, labels_all)

# ── Entraînement ─────────────────────────────────────────────────
model = ResNet18CBAM().to(DEVICE)
crit  = nn.CrossEntropyLoss(weight=w)
opt   = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = CosineAnnealingLR(opt, T_max=NUM_EPOCHS, eta_min=LR*0.01)

best_f1, best_ep, patience_cnt = 0.0, 0, 0
best_path = os.path.join(OUTPUT_DIR, "task3_best_model_v3.pth")

print(f"\n🚀 Entraînement v3 — LR={LR}, dropout=0.5, poids intermédiaires")
print("-"*60)

for ep in range(1, NUM_EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_acc, tr_rec             = train_epoch(model, train_dl, opt, crit, DEVICE)
    vl_loss, vl_acc, vl_rec, vl_f1,_,_ = eval_epoch(model, val_dl, crit, DEVICE)
    sched.step()
    print(f"Ep {ep:3d} | "
          f"Tr Loss={tr_loss:.4f} Acc={tr_acc:.3f} Rec={tr_rec:.3f} | "
          f"Val Loss={vl_loss:.4f} Acc={vl_acc:.3f} Rec={vl_rec:.3f} F1={vl_f1:.3f} | "
          f"{time.time()-t0:.0f}s")
    if vl_f1 > best_f1:
        best_f1, best_ep, patience_cnt = vl_f1, ep, 0
        torch.save({'epoch':ep, 'model_state_dict':model.state_dict(),
                    'val_f1':vl_f1, 'val_acc':vl_acc, 'val_rec':vl_rec}, best_path)
        print(f"  ✅ Sauvegardé (F1={vl_f1:.4f})")
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"\n⏹️  Early stopping epoch {ep}"); break

# ── Test final ───────────────────────────────────────────────────
model.load_state_dict(torch.load(best_path, map_location=DEVICE)['model_state_dict'])
_, t_acc, t_rec, t_f1, t_preds, t_labels = eval_epoch(model, test_dl, crit, DEVICE)

print("\n" + "="*60)
print("RÉSULTATS FINAUX v3")
print("="*60)
print(f"Accuracy : {t_acc*100:.2f}%  {'✅' if t_acc>=0.90 else '❌'}  [objectif >90%]")
print(f"Recall   : {t_rec*100:.2f}%  {'✅' if t_rec>=0.85 else '❌'}  [objectif >85%]")
print(f"F1-Score : {t_f1:.4f}")
print()
print(classification_report(t_labels, t_preds,
                             target_names=['Non détectable','Détectable']))
cm = confusion_matrix(t_labels, t_preds)
print(f"Matrice confusion : TN={cm[0,0]} FP={cm[0,1]} | FN={cm[1,0]} TP={cm[1,1]}")
print(f"\n💾 Modèle : {best_path}")

✅ 4715 samples chargés instantanément
Poids : [w0=1.50, w1=0.70]  ← intermédiaires

🚀 Entraînement v3 — LR=3e-05, dropout=0.5, poids intermédiaires
------------------------------------------------------------
Ep   1 | Tr Loss=0.6700 Acc=0.426 Rec=0.079 | Val Loss=0.6531 Acc=0.420 Rec=0.039 F1=0.074 | 4s
  ✅ Sauvegardé (F1=0.0743)
Ep   2 | Tr Loss=0.6357 Acc=0.535 Rec=0.303 | Val Loss=0.6165 Acc=0.538 Rec=0.319 F1=0.453 | 4s
  ✅ Sauvegardé (F1=0.4528)
Ep   3 | Tr Loss=0.5973 Acc=0.599 Rec=0.450 | Val Loss=0.5854 Acc=0.699 Rec=0.695 F1=0.735 | 4s
  ✅ Sauvegardé (F1=0.7350)
Ep   4 | Tr Loss=0.5628 Acc=0.663 Rec=0.579 | Val Loss=0.5539 Acc=0.708 Rec=0.673 F1=0.734 | 4s
Ep   5 | Tr Loss=0.5250 Acc=0.704 Rec=0.646 | Val Loss=0.5257 Acc=0.756 Rec=0.834 F1=0.804 | 4s
  ✅ Sauvegardé (F1=0.8037)
Ep   6 | Tr Loss=0.4970 Acc=0.739 Rec=0.695 | Val Loss=0.4942 Acc=0.769 Rec=0.831 F1=0.812 | 4s
  ✅ Sauvegardé (F1=0.8119)
Ep   7 | Tr Loss=0.4694 Acc=0.762 Rec=0.730 | Val Loss=0.4901 Acc=0.776 Rec=0.86

In [ ]:
import os, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report, confusion_matrix
from torchvision.models import efficientnet_b0
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = "/content/drive/MyDrive/skipper_task3/task3_outputs"
DEVICE     = torch.device("cuda")
BATCH_SIZE = 64
NUM_EPOCHS = 50
LR         = 3e-4
PATIENCE   = 10

# ── Chargement instantané ────────────────────────────────────────
ckpt     = torch.load("/content/drive/MyDrive/skipper_task3/preprocessed_data.pt")
X_tensor = ckpt['X']
y_tensor = ckpt['y']
print(f"✅ {X_tensor.shape[0]} samples chargés instantanément")

indices = np.arange(len(y_tensor))
all_y   = y_tensor.numpy()
idx_trval, idx_test  = train_test_split(indices, test_size=0.15,
                                         stratify=all_y, random_state=42)
idx_train, idx_val   = train_test_split(idx_trval, test_size=0.15,
                                         stratify=all_y[idx_trval], random_state=42)

train_dl = DataLoader(TensorDataset(X_tensor[idx_train], y_tensor[idx_train]),
                      batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(TensorDataset(X_tensor[idx_val],   y_tensor[idx_val]),
                      batch_size=BATCH_SIZE, shuffle=False)
test_dl  = DataLoader(TensorDataset(X_tensor[idx_test],  y_tensor[idx_test]),
                      batch_size=BATCH_SIZE, shuffle=False)

# ── Poids — on repart sur la v1 qui avait le meilleur accuracy ──
w = torch.tensor([1.25, 0.83], dtype=torch.float32).to(DEVICE)
print(f"Poids : [w0={w[0]:.2f}, w1={w[1]:.2f}]")

# ── EfficientNet-B0 adapté 4 canaux ─────────────────────────────
class EfficientNet4ch(nn.Module):
    """
    EfficientNet-B0 adapté pour 4 canaux d'entrée.
    On remplace la première conv 3ch→32 par une conv 4ch→32.
    On garde tous les poids pré-entraînés sauf cette première couche.
    """
    def __init__(self, n_cls=2, drop=0.3):
        super().__init__()
        # Charger EfficientNet-B0 pré-entraîné
        base = efficientnet_b0(weights='IMAGENET1K_V1')

        # Remplacer la première conv : 3ch → 4ch
        old_conv = base.features[0][0]
        new_conv = nn.Conv2d(
            in_channels=4,
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False
        )
        # Initialisation intelligente : copier les poids RGB
        # et initialiser le 4ème canal comme moyenne des 3 premiers
        with torch.no_grad():
            new_conv.weight[:, :3, :, :] = old_conv.weight
            new_conv.weight[:, 3:4, :, :] = old_conv.weight.mean(dim=1, keepdim=True)

        base.features[0][0] = new_conv

        # Remplacer la tête de classification
        in_features = base.classifier[1].in_features
        base.classifier = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, n_cls)
        )

        self.model = base

    def forward(self, x):
        return self.model(x)

def augment_batch(x):
    mask_h = torch.rand(x.size(0), device=x.device) > 0.5
    x[mask_h] = x[mask_h].flip(-1)
    mask_v = torch.rand(x.size(0), device=x.device) > 0.5
    x[mask_v] = x[mask_v].flip(-2)
    if torch.rand(1) > 0.7:
        x += torch.randn_like(x) * 0.05
    return x

def train_epoch(model, loader, opt, crit, device):
    model.train()
    loss_sum, preds_all, labels_all = 0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        x = augment_batch(x)
        opt.zero_grad()
        out  = model(x)
        loss = crit(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        loss_sum += loss.item()
        preds_all.extend(out.argmax(1).detach().cpu().numpy())
        labels_all.extend(y.cpu().numpy())
    n = len(loader)
    return (loss_sum/n,
            accuracy_score(labels_all, preds_all),
            recall_score(labels_all, preds_all, zero_division=0))

@torch.no_grad()
def eval_epoch(model, loader, crit, device):
    model.eval()
    loss_sum, preds_all, labels_all = 0, [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out  = model(x)
        loss_sum += crit(out, y).item()
        preds_all.extend(out.argmax(1).cpu().numpy())
        labels_all.extend(y.cpu().numpy())
    n = len(loader)
    return (loss_sum/n,
            accuracy_score(labels_all, preds_all),
            recall_score(labels_all, preds_all, zero_division=0),
            f1_score(labels_all, preds_all, zero_division=0),
            preds_all, labels_all)

# ── Stratégie d'entraînement en 2 phases ────────────────────────
# Phase 1 : geler le backbone, entraîner seulement la tête (5 epochs)
# Phase 2 : dégeler tout, fine-tuning complet (epochs restantes)
# Ça évite de détruire les poids pré-entraînés au début

model = EfficientNet4ch().to(DEVICE)
total = sum(p.numel() for p in model.parameters())
print(f"Paramètres : {total:,}  (EfficientNet-B0 = 5.3M vs ResNet18 = 11.3M)")

crit = nn.CrossEntropyLoss(weight=w)
best_path = os.path.join(OUTPUT_DIR, "task3_best_model_v4_efficientnet.pth")

print(f"\n📌 PHASE 1 — Chauffe de la tête (5 epochs, backbone gelé)")
print("-"*60)

# Geler tout sauf la tête
for name, param in model.named_parameters():
    if 'classifier' not in name and 'features.0' not in name:
        param.requires_grad = False

opt_phase1 = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3, weight_decay=1e-4
)

best_f1, best_ep, patience_cnt = 0.0, 0, 0

for ep in range(1, 6):
    t0 = time.time()
    tr_loss, tr_acc, tr_rec             = train_epoch(model, train_dl, opt_phase1, crit, DEVICE)
    vl_loss, vl_acc, vl_rec, vl_f1,_,_ = eval_epoch(model, val_dl, crit, DEVICE)
    print(f"Ph1 Ep {ep} | Tr Acc={tr_acc:.3f} Rec={tr_rec:.3f} | "
          f"Val Acc={vl_acc:.3f} Rec={vl_rec:.3f} F1={vl_f1:.3f} | {time.time()-t0:.0f}s")
    if vl_f1 > best_f1:
        best_f1, best_ep = vl_f1, ep
        torch.save({'epoch':ep, 'model_state_dict':model.state_dict(),
                    'val_f1':vl_f1}, best_path)
        print(f"  ✅ Sauvegardé (F1={vl_f1:.4f})")

print(f"\n📌 PHASE 2 — Fine-tuning complet ({NUM_EPOCHS} epochs, tout dégelé)")
print("-"*60)

# Dégeler tout le réseau
for param in model.parameters():
    param.requires_grad = True

opt_phase2 = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = CosineAnnealingLR(opt_phase2, T_max=NUM_EPOCHS, eta_min=LR*0.01)
patience_cnt = 0

for ep in range(1, NUM_EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_acc, tr_rec             = train_epoch(model, train_dl, opt_phase2, crit, DEVICE)
    vl_loss, vl_acc, vl_rec, vl_f1,_,_ = eval_epoch(model, val_dl, crit, DEVICE)
    sched.step()
    print(f"Ep {ep:3d} | Tr Loss={tr_loss:.4f} Acc={tr_acc:.3f} Rec={tr_rec:.3f} | "
          f"Val Loss={vl_loss:.4f} Acc={vl_acc:.3f} Rec={vl_rec:.3f} F1={vl_f1:.3f} | "
          f"{time.time()-t0:.0f}s")
    if vl_f1 > best_f1:
        best_f1, best_ep, patience_cnt = vl_f1, ep, 0
        torch.save({'epoch':ep, 'model_state_dict':model.state_dict(),
                    'val_f1':vl_f1, 'val_acc':vl_acc, 'val_rec':vl_rec}, best_path)
        print(f"  ✅ Sauvegardé (F1={vl_f1:.4f})")
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"\n⏹️  Early stopping epoch {ep}"); break

# ── Test final ───────────────────────────────────────────────────
print(f"\n🏆 Meilleur : Epoch {best_ep}  F1={best_f1:.4f}")
model.load_state_dict(torch.load(best_path, map_location=DEVICE)['model_state_dict'])
_, t_acc, t_rec, t_f1, t_preds, t_labels = eval_epoch(model, test_dl, crit, DEVICE)

print("\n" + "="*60)
print("RÉSULTATS FINAUX v4 — EfficientNet-B0")
print("="*60)
print(f"Accuracy : {t_acc*100:.2f}%  {'✅' if t_acc>=0.90 else '❌'}  [objectif >90%]")
print(f"Recall   : {t_rec*100:.2f}%  {'✅' if t_rec>=0.85 else '❌'}  [objectif >85%]")
print(f"F1-Score : {t_f1:.4f}")
print()
print(classification_report(t_labels, t_preds,
                             target_names=['Non détectable','Détectable']))
cm = confusion_matrix(t_labels, t_preds)
print(f"Matrice confusion : TN={cm[0,0]} FP={cm[0,1]} | FN={cm[1,0]} TP={cm[1,1]}")
print(f"\n💾 Modèle : {best_path}")

✅ 4715 samples chargés instantanément
Poids : [w0=1.25, w1=0.83]
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 224MB/s]


Paramètres : 4,336,286  (EfficientNet-B0 = 5.3M vs ResNet18 = 11.3M)

📌 PHASE 1 — Chauffe de la tête (5 epochs, backbone gelé)
------------------------------------------------------------
Ph1 Ep 1 | Tr Acc=0.671 Rec=0.680 | Val Acc=0.743 Rec=0.850 F1=0.798 | 6s
  ✅ Sauvegardé (F1=0.7984)
Ph1 Ep 2 | Tr Acc=0.730 Rec=0.747 | Val Acc=0.669 Rec=0.643 F1=0.700 | 6s
Ph1 Ep 3 | Tr Acc=0.744 Rec=0.761 | Val Acc=0.698 Rec=0.654 F1=0.722 | 6s
Ph1 Ep 4 | Tr Acc=0.761 Rec=0.795 | Val Acc=0.674 Rec=0.562 F1=0.674 | 6s
Ph1 Ep 5 | Tr Acc=0.759 Rec=0.767 | Val Acc=0.679 Rec=0.704 F1=0.725 | 6s

📌 PHASE 2 — Fine-tuning complet (50 epochs, tout dégelé)
------------------------------------------------------------
Ep   1 | Tr Loss=0.4822 Acc=0.781 Rec=0.792 | Val Loss=0.5278 Acc=0.779 Rec=0.939 F1=0.836 | 6s
  ✅ Sauvegardé (F1=0.8360)
Ep   2 | Tr Loss=0.3689 Acc=0.838 Rec=0.869 | Val Loss=0.4142 Acc=0.809 Rec=0.806 F1=0.835 | 6s
Ep   3 | Tr Loss=0.3326 Acc=0.850 Rec=0.850 | Val Loss=0.4032 Acc=0.786 Rec=0

In [ ]:
torch.save({'X': X_tensor, 'y': y_tensor},
           "/content/drive/MyDrive/skipper_task3/preprocessed_data.pt")
print("✅ Données sauvegardées pour la prochaine fois")

✅ Données sauvegardées pour la prochaine fois
